In [1]:
%load_ext autoreload
%autoreload 2

import os
import xarray as xr
import dask
import numpy as np
import pandas as pd

from aurora_benchmark.utils import Statistics
from aurora_benchmark.plots import (
    rmse_curves, 
    signed_difference_maps, 
    prediction_maps, 
    find_closest_files
)

np.random.seed(0)

dask.config.set(scheduler='threads')

In [2]:
# Function arguments
era5_surface_paths = [
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/10v_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/2t_2021-2022-6h-1440x721.nc",
#  - data/era5_wb2/2021-2022-6h-1440x721/tp_2021-2022-6h-1440x721.nc # original only
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/10u_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/msl_2021-2022-6h-1440x721.nc",
#  - data/era5_wb2/2021-2022-6h-1440x721/sst_2021-2022-6h-1440x721.nc # original only
]
era5_atmospheric_paths = [
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/q_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/t_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/u_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/v_2021-2022-6h-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/z_2021-2022-6h-1440x721.nc",
]
era5_static_paths = [
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/lsm_static-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/z_static-1440x721.nc",
"/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2/2021-2022-6h-1440x721/slt_static-1440x721.nc",
]

forecast_dir = "/projects/prjs0981/ewalt/aurora_benchmark/data/era5_wb2_forecasts/2021-2022-6h-1w-6w-1440x721_original_variables_parallel/"
eval_dir = "../figures/era5_wb2_eval/2021-2022-6h-1w-6w-1440x721_original_variables_parallel/"

# Load the data into a single dataset with the same coords but multiple variables
surface_ds = xr.merge([
    xr.open_dataset(p, engine="netcdf4", chunks={"time": 50, "latitude": 721, "longitude": 1440})
    for p in era5_surface_paths
])

atmospheric_ds = xr.merge([
    xr.open_dataset(p, engine="netcdf4", chunks={"time": 50, "latitude": 721, "longitude": 1440, "level": 1})
    for p in era5_atmospheric_paths
])

static_ds = xr.merge([
    xr.open_dataset(p, engine="netcdf4", chunks={"latitude": 721, "longitude": 1440})
    for p in era5_static_paths
])

assert os.path.exists(forecast_dir), f"Forecast directory {forecast_dir} does not exist"
os.makedirs(eval_dir, exist_ok=True)

surface_ds.dims, atmospheric_ds.dims, static_ds.dims

/home/ewalt/.local/lib/python3.11/site-packages/xarray/core/dataset.py:273: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/home/ewalt/.local/lib/python3.11/site-packages/xarray/core/dataset.py:273: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/home/ewalt/.local/lib/python3.11/site-packages/xarray/core/dataset.py:273: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 50. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/home/ewalt/.local/lib/python3.11/site-packages/xarray/core/dataset.py:273: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at ind

(FrozenMappingWarningOnValuesAccess({'latitude': 721, 'longitude': 1440, 'time': 1460}),
 FrozenMappingWarningOnValuesAccess({'latitude': 721, 'level': 13, 'longitude': 1440, 'time': 1460}),
 FrozenMappingWarningOnValuesAccess({'latitude': 721, 'longitude': 1440}))

In [3]:
global_statistics = {
    "surface_vars": Statistics(),
    "atmospheric_vars": Statistics(),
}
med_statistics = {
    "surface_vars": Statistics(),
    "atmospheric_vars": Statistics(),
}
med_dry_statistics = {
    "surface_vars": Statistics(),
    "atmospheric_vars": Statistics(),
}
med_wet_statistics = {
    "surface_vars": Statistics(),
    "atmospheric_vars": Statistics(),
}

# define wet (oct-mar) and dry (apr-sep) seasons
wet_months = [10, 11, 12, 1, 2, 3]
dry_months = [4, 5, 6, 7, 8, 9]

# define med
med_region = {    
    "latitude": slice(47, 29), 
    "longitude": slice(-8, 38) 
}

# loop over files
for i, file in enumerate(os.scandir(forecast_dir)):
    # parse file info
    file_info = file.name.replace(".nc", "").split("_")
    variable_name = file_info[1]
    file_info = file_info[2].split("-")
    init_time = pd.Timestamp(file_info[0])
    base_frequency = file_info[1]
    eval_aggregation = file_info[2]
    eval_start = file_info[3]
    forecast_horizon = file_info[4]    

    # feedback
    print(f"Processing {file.name}")

    # load forecast
    pred_trajectory = xr.open_dataset(file.path, engine="netcdf4")
    assert pd.Timedelta((pred_trajectory.lead_time[1]-pred_trajectory.lead_time[0]).values) == pd.Timedelta(eval_aggregation)

    # load ERA5 gt from surface_ds and atmospheric_ds
    true_ds = atmospheric_ds if variable_name in atmospheric_ds.data_vars else surface_ds
    true_trajectory = true_ds[variable_name]\
            .sel(time=slice(init_time+pd.Timedelta(eval_start), init_time+pd.Timedelta(forecast_horizon)))
            

    # resample gt to eval_aggregation
    true_trajectory = true_trajectory.resample(time=pd.Timedelta(eval_aggregation), origin=init_time).mean()
    assert pd.Timedelta((true_trajectory.time[1]-true_trajectory.time[0]).values) == pd.Timedelta(eval_aggregation)
    
    # get the index of times that are in wet and dry months respectively
    dry_times = (init_time + pred_trajectory.lead_time).dt.month.isin(dry_months)
    wet_times = (init_time + pred_trajectory.lead_time).dt.month.isin(wet_months)
    
    # rename true time to lead time
    true_trajectory = true_trajectory.rename({"time": "lead_time"})
    true_trajectory["lead_time"] = true_trajectory["lead_time"] - np.datetime64(init_time)

    # shape
    sizes = pred_trajectory.sizes
    nlt = len(np.unique(pred_trajectory.lead_time.values))
    if variable_name in atmospheric_ds.data_vars:
        stat_key = "atmospheric_vars"
    else:
        stat_key = "surface_vars"
        
    # compute signed error
    signed_error_ds = (pred_trajectory - true_trajectory)
    signed_error_ds = signed_error_ds.assign_coords({"longitude": signed_error_ds.longitude.values-180, "time": init_time})

    # accumulate statistics
    global_statistics[stat_key].update(signed_error_ds)
    med_statistics[stat_key].update(signed_error_ds.sel(med_region))    
    
    # accumulate seasonal statistics
    # !!! We only take the ones that are FULLY in the season (otherwise we cannot concatenate...)
    if dry_times.all().item():
        med_dry_statistics[stat_key].update(signed_error_ds.sel(med_region).isel(lead_time=dry_times))
    if wet_times.all().item():
        med_wet_statistics[stat_key].update(signed_error_ds.sel(med_region).isel(lead_time=wet_times))

Processing forecast_10v_20211105T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_u_20210910T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20210716T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_msl_20210611T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20210910T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_2t_20210305T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20211029T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_u_20210514T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20211105T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_10v_20210521T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_u_20210122T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20210101T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_u_20210219T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_t_20210827T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_t_20210212T060000-6h-1w-1w-6w-1440x720.nc
Processing forecast_v_20211001T060000-6h-1w-1w-6w-1440x720.nc
P

In [ ]:
wet_indexes, wet_files = find_closest_files(forecast_dir, 
                                            pd.Timestamp("2021-11-15"),
                                            variables=["2t", "msl", "z"])
dry_indexes, dry_files = find_closest_files(forecast_dir, 
                                            pd.Timestamp("2021-06-10"),
                                            variables=["2t", "msl", "z"])

for indexes in [wet_indexes, dry_indexes]:
    for index in indexes:
        prediction_maps(
            forecast_dir,
            atmospheric_ds,
            surface_ds,
            file_index=index,
            eval_dir=eval_dir,
            lead_times=None,#[pd.Timedelta(f"{i}w") for i in range(1, 7)],
            level=250
        )

In [ ]:
rmse_curves(
    global_statistics,
    med_statistics,
    med_wet_statistics,
    med_dry_statistics,
    fig_title=f"RMSE for base_frequency={base_frequency}, eval_aggregation={eval_aggregation}, eval_start={eval_start}, forecast_horizon={forecast_horizon}",
    std_fig_title=f"RMSE (with std shading) for base_frequency={base_frequency}, eval_aggregation={eval_aggregation}, eval_start={eval_start}, forecast_horizon={forecast_horizon}",
    eval_dir=eval_dir,
    nrows=4,
    std_plot=True
)

In [ ]:
signed_difference_maps(
    global_statistics,
    med_statistics,
    med_wet_statistics,
    med_dry_statistics,
    eval_dir=eval_dir,
    variables=["2t", "msl", "z_250"]
)